    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 2b: Implement a program which, given (a) a query imageID or image file and 
    (b) positive integer k, identifies and lists k most likely matching labels, along 
    with their scores, under the RESNET50 neural network model.

In [25]:
# Import Libraries 
import pandas as pd
import numpy as np
import torch
import torchvision
import PIL
from pathlib import Path
from tqdm import tqdm
import scipy
import sklearn.cluster
import math
# Custom Functions
from utils.query_input_processor import get_query_image
from utils.image_utils import convert_to_rgb

In [26]:
# Research References:
# https://saturncloud.io/blog/how-to-check-if-pytorch-is-using-the-gpu/
# https://saturncloud.io/blog/how-to-train-a-pytorch-model-on-gpu/
# https://docs.scipy.org/doc/scipy/reference/spatial.distance.html
# https://learnopencv.com/image-classification-using-transfer-learning-in-pytorch/
# https://pytorch.org/docs/master/generated/torch.topk.html
# https://pytorch.org/vision/stable/models.html

In [27]:
# These utility functions are unique to task 2b
# Using convert_to_rgb, get_query_image from utility folder

# These function gets the device to use as the goal of 
# the code is to use GPU if possible
def getDevice():
    torchDevice = "cpu"
    if torch.cuda.is_available():
        torchDevice = "cuda"

    print("Using Device: ", torchDevice)
    return torchDevice


# This function converts the image to 224x224 as required by
# the ResNet model, then transforms it with the ResNet weight,
# and then gets the ResNet Input
def resNetImageTransformer(image, resNetModel, resNetWeight, torchDevice):
    # Convert Non-RGB to RGB
    if image.mode != "RGB":
        image = convert_to_rgb(image)
        
    # Resize Image
    imageCopy = image.resize(size=(224, 224))
    
    # Put image into model and get results
    imageTransform = resNetWeight.transforms()(imageCopy).unsqueeze(0).to(torchDevice)
    result = resNetModel(imageTransform).squeeze(0).softmax(0).to(torchDevice)

    return result.detach().cpu().numpy()


# This function gets the name of the label id given in the database
def getCalTechCategory(caltechDB, label_id):
    return caltechDB.annotation_categories[label_id]

# This function setsup the ResNet Model
def getResNetModel():
    resNetWeight = torchvision.models.ResNet50_Weights.DEFAULT
    resNetModel = torchvision.models.resnet50(progress=True, weights=resNetWeight).to(torchDevice)
    for parameter in resNetModel.parameters():
        parameter.requires_grad = False
    resNetModel.zero_grad(set_to_none=True)
    resNetModel.eval()

    return resNetWeight, resNetModel


In [28]:
# Function takes an image and finds the k similar labels using kmeans
def kSimilarImages_2b(image, df, caltechDB, k, resNetWeight, resNetModel):
    # Clustering and Parameters used from: https://www.analyticsvidhya.com/blog/2021/01/a-simple-guide-to-centroid-based-clustering-with-python-code/

    # Get training set of dataset
    traindf = df[df["ImageID"] % 2 == 0]

    # Extract ResNet Reuslts of imageID
    result = resNetImageTransformer(image, resNetWeight, torchDevice)

    # Create the KMeans Clusters and get the centroids
    kmeans = sklearn.cluster.KMeans(n_clusters=101, init="k-means++", random_state=0, n_init="auto").fit([list(row.astype(float)) for row in traindf["ResNet"]])
    # kmeans = sklearn.cluster.DBSCAN(eps=3, min_samples=2).fit([list(row.astype(float)) for row in traindf["ResNet"]])
    # print(kmeans.cluster_centers_)
    # print(set(kmeans.labels_))

    # Go through the centroids and find the closest ones
    counter = 0
    df_list = []

    for centroid in kmeans.cluster_centers_:
        # Distance is affecting results
        distance = scipy.spatial.distance.cosine(result, centroid)
        # if counter != caltechDB[imageID][1]: # I was thinking about comparing the image's label (centroid) to find nearest ones
        #     distance = scipy.spatial.distance.cityblock(kmeans.cluster_centers_[caltechDB[imageID][1]], centroid)
        #     df_list.append({"LabelID": counter, "Label": getCalTechCategory(caltechDB, counter), "Distance": distance})
        df_list.append({"LabelID": counter, "Label": getCalTechCategory(caltechDB, counter), "Distance": distance})
        counter = counter + 1
    
    # Sort the results and get the top k results
    distanceDF = pd.DataFrame(df_list)
    results = distanceDF.sort_values(by="Distance")[0:k].reset_index()[["LabelID", "Label", "Distance"]]

    print(results)

In [29]:
# This function is the code to generate the resnet
# vectors and exports the database so it can be called in k similar labels
# As a note, task2b and k similar label functions are two differet functions so
# I could test the k similar labels without waiting for the resnet model each time to 
# analyze the images.
def task2b(caltechDB):
    # Create Pandas Dataframe
    df_list = []

    # For Loop in databases
    for i in tqdm(range(0, len(caltechDB)), desc="Extracting Features", ncols=100):
        # for i in range(1580, 1582, 1):
        # Get Image
        image = caltechDB[i][0]

        # Put image into model and get results
        result = resNetImageTransformer(image, resNetModel, resNetWeight, torchDevice)

        # Attach to List
        row_df = {
            "ImageID": i,
            "Label": getCalTechCategory(caltechDB, caltechDB[i][1]),
            "LabelID": caltechDB[i][1],
            "ResNet": result,
        }
        df_list.append(row_df)

    torch.cuda.empty_cache()

    # Convert List to DataFrame for easy accessability
    df = pd.DataFrame(df_list)
    # print(df.head(5))

    df.to_pickle("./testFile_2b")

In [30]:
# This function gets the reuqired user input for the functions
def userInput():
    IMAGE_INPUT = input(
    """
    Provide one of the following:
    1. An Image ID in the Caltech101 dataset in range [0, 8676].
    2. The name of an image file in /Code/input/ directory (eg: image.jpg).
    """
    )

    kLabels = int(input("Enter K, the number of similar labels to find for ResNet model."))

    image = get_query_image(IMAGE_INPUT)
    print("User inputed image ID / image: ", IMAGE_INPUT)
    display(image)
    return image, kLabels



In [31]:
# Main code that includes the user input, then calls task2b, then the k similar labels function

# Load Database
caltechDB = torchvision.datasets.Caltech101("./Databases/CaltechDB/", download=True, target_type="category")

# Get Device
torchDevice = getDevice()

# Get ResNet Modle
resNetWeight, resNetModel = getResNetModel(torchDevice)

# Get User Inputs
image, kLabels = userInput()

# Get the vectors for the images
task2b(caltechDB)

# Get K Similar Images based on an image
df = pd.read_pickle("./testFile_2b")
kSimilarImages_2b(image, df, caltechDB, kLabels, resNetWeight, resNetModel)

Files already downloaded and verified


NameError: name 'torchDevice' is not defined